In [ ]:
# 1. Check the GPU: all team runs use the same GPU type (L4)
import torch
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, f"{gpu.total_memory / 2**30:.1f} GiB")
assert "L4" in gpu.name, "Switch the runtime to an L4 GPU (Runtime > Change runtime type)"

In [ ]:
# 2. Get a fresh copy of main
%cd /content
!rm -rf kv-evidence-loss
!git clone -q https://github.com/burakzor/kv-evidence-loss.git
%cd /content/kv-evidence-loss
!git log -1 --format="%h %ad %s" --date=short

In [ ]:
# 3. Install the pinned team environment (torch is Colab's own CUDA build)
!pip install -q -r requirements.txt
!pip list 2>/dev/null | grep -iE "^(torch|transformers|tokenizers|accelerate|datasets|pandas|numpy|pyarrow|tqdm) "

In [ ]:
# 4. Record the full environment of this run (GPU, driver, Python, CUDA, repository commit, every package); both budgets come from the same run
!mkdir -p results/answers
!(nvidia-smi --query-gpu=name,driver_version --format=csv,noheader; python --version; python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"; echo "commit $(git rev-parse --short HEAD)"; pip freeze) > results/answers/qrhead_0.2_env.txt
!cp results/answers/qrhead_0.2_env.txt results/answers/qrhead_0.1_env.txt
!head -5 results/answers/qrhead_0.2_env.txt

In [ ]:
# 5. Optional: Hugging Face token for a faster download (only works in the Colab web UI)
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print(f"No HF token ({type(e).__name__}); downloading anonymously.")

In [ ]:
# 6. First run on 3 questions: answers to /content/qrhead_test, masks to results/raw/kept_limit3 (the team's test-run folder), so that nothing mixes with the full run
# (first run downloads Qwen3-8B, ~16 GB)
!cd scripts && python -c "from pathlib import Path; import qrhead_run as q; q.run(q.MODEL_NAME, q.HEADS_PATH, 3, Path('/content/qrhead_test/answers'), q.ROOT / 'results' / 'raw' / 'kept_limit3' / 'qrhead')"

In [ ]:
# 7. Full run: score the 10 paragraphs of all 169 clean questions with the QRHead heads, answer at budgets 0.2 and 0.1
!python scripts/qrhead_run.py

In [ ]:
# 8. Look at the results per budget, next to the full-cache baseline
import pandas as pd
full_df = pd.read_parquet("results/answers/full_1.0.parquet")
summary_rows = []
for budget in [0.2, 0.1]:
    df = pd.read_parquet(f"results/answers/qrhead_{budget}.parquet")
    summary_rows.append({
        "budget": budget,
        "questions": len(df),
        "contains_alias": df["contains_alias"].mean(),
        "paragraphs_kept": df["n_paragraphs_kept"].mean(),
        "prompt_kept_fraction": df["prompt_kept_fraction"].mean(),
        "a_kept": df["a_kept"].mean(),
        "b_kept": df["b_kept"].mean(),
        "both_kept": (df["a_kept"] & df["b_kept"]).mean(),
        "rank_s": df["rank_s"].mean(),
        "generate_s": df["generate_s"].mean(),
        "peak_mem_gb": df["peak_mem_gb"].max(),
    })
print("full cache contains_alias:", round(full_df["contains_alias"].mean(), 3))
print(pd.DataFrame(summary_rows).round(3).to_string(index=False))
df = pd.read_parquet("results/answers/qrhead_0.1.parquet")
print(df["n_paragraphs_kept"].value_counts().sort_index())

In [ ]:
# 9. Zip the kept masks (paths inside the zip: qrhead/{budget}/{id}.npz, unzip into results/raw/kept/) and copy them to Google Drive (the Colab disk is deleted when the session ends)
!ls results/raw/kept/qrhead/0.2 | wc -l
!ls results/raw/kept/qrhead/0.1 | wc -l
!cd results/raw/kept && zip -qr /content/qrhead_masks.zip qrhead
!ls -lh /content/qrhead_masks.zip
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/kv-evidence-loss/raw/kept
!cp -r results/raw/kept/qrhead /content/drive/MyDrive/kv-evidence-loss/raw/kept/
!cp /content/qrhead_masks.zip /content/drive/MyDrive/kv-evidence-loss/raw/

In [ ]:
# 10. Download the answers tables, the run settings, the environment records and the zipped masks
from google.colab import files
for budget in [0.2, 0.1]:
    files.download(f"results/answers/qrhead_{budget}.parquet")
    files.download(f"results/answers/qrhead_{budget}_config.json")
    files.download(f"results/answers/qrhead_{budget}_env.txt")
files.download("/content/qrhead_masks.zip")